# Washout-corrected tiny-paper outputs

This notebook creates compact tables and figures using a 12-week
decision horizon and a six-week post-decision evaluation washout.

## 1. Imports and paths

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in [
            CURRENT_DIR,
            CURRENT_DIR.parent,
        ]
        if (
            root
            / "results"
            / "tables"
        ).is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from {CURRENT_DIR}"
    )

TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
)
PAPER_DIR = (
    PROJECT_ROOT
    / "paper"
    / "generated_v3"
)
PAPER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CALIBRATION_PATH = (
    TABLE_DIR
    / "product_stockpiling_calibration_summary_v2.csv"
)
HOLDOUT_PATH = (
    TABLE_DIR
    / "product_holdout_event_validation_v2.csv"
)
ACTION_PATH = (
    TABLE_DIR
    / "product_supported_action_clusters.csv"
)
ESCALATION_PATH = (
    TABLE_DIR
    / "product_escalation_thresholds_washout.csv"
)
CATEGORY_SUMMARY_PATH = (
    TABLE_DIR
    / "category_capacity_summary_washout6.csv"
)
CATEGORY_COMPARISON_PATH = (
    TABLE_DIR
    / "category_capacity_comparisons_washout6.csv"
)
CATEGORY_CALENDAR_PATH = (
    TABLE_DIR
    / "category_capacity_calendars_washout6.csv"
)
WASHOUT_SENSITIVITY_PATH = (
    TABLE_DIR
    / "category_washout_sensitivity.csv"
)

MAIN_ALPHA = 2.24
MAIN_CAPACITY = 2
PRIMARY_GRID = (
    "empirical_w6"
)

## 2. Load outputs

In [ ]:
required = [
    CALIBRATION_PATH,
    HOLDOUT_PATH,
    ACTION_PATH,
    ESCALATION_PATH,
    CATEGORY_SUMMARY_PATH,
    CATEGORY_COMPARISON_PATH,
    CATEGORY_CALENDAR_PATH,
    WASHOUT_SENSITIVITY_PATH,
]

missing = [
    path
    for path in (
        required
    )
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        "Run notebooks 07b, 08c, and 09c first. Missing: "
        + ", ".join(
            str(
                path
            )
            for path in (
                missing
            )
        )
    )

calibration = pd.read_csv(
    CALIBRATION_PATH
)
holdout = pd.read_csv(
    HOLDOUT_PATH
)
support = pd.read_csv(
    ACTION_PATH
)
escalation = pd.read_csv(
    ESCALATION_PATH
)
category_summary = pd.read_csv(
    CATEGORY_SUMMARY_PATH
)
category_comparison = pd.read_csv(
    CATEGORY_COMPARISON_PATH
)
category_calendar = pd.read_csv(
    CATEGORY_CALENDAR_PATH
)
washout_sensitivity = pd.read_csv(
    WASHOUT_SENSITIVITY_PATH
)

## 3. Final tables

In [ ]:
calibration_table = (
    calibration[
        [
            "upc",
            "product_name",
            "isolated_events",
            "shrinkage_reliability",
            "price_elasticity_median",
            "promotion_lift_median",
            "displacement_median",
            "persistence_median",
        ]
    ]
    .sort_values(
        "displacement_median",
        ascending=False,
    )
)

action_table = (
    support.loc[
        support[
            "selected_for_grid"
        ]
    ][
        [
            "upc",
            "product_name",
            "action",
            "support_count",
            "support_panels",
        ]
    ]
    .sort_values(
        [
            "upc",
            "action",
        ]
    )
)

escalation_table = (
    escalation.loc[
        escalation[
            "grid_name"
        ].eq(
            PRIMARY_GRID
        )
    ][
        [
            "upc",
            "product_name",
            "alpha_1_promotions",
            "alpha_2_promotions",
            "alpha_4_promotions",
        ]
    ]
)

main_comparison = (
    category_comparison.loc[
        category_comparison[
            "weekly_capacity"
        ].eq(
            MAIN_CAPACITY
        )
        & np.isclose(
            category_comparison[
                "alpha"
            ],
            MAIN_ALPHA,
        )
    ]
    .sort_values(
        "grid_name"
    )
)

capacity_summary = (
    category_comparison.groupby(
        [
            "grid_name",
            "weekly_capacity",
        ],
        observed=True,
    )
    .agg(
        mean_VDO=(
            "value_of_dynamic_optimization",
            "mean",
        ),
        maximum_VDO=(
            "value_of_dynamic_optimization",
            "max",
        ),
        active_set_change_share=(
            "active_set_changed",
            "mean",
        ),
        mean_timing_disagreement=(
            "product_week_timing_disagreement",
            "mean",
        ),
        mean_dynamic_supported_share=(
            "dynamic_supported_share",
            "mean",
        ),
    )
    .reset_index()
)

washout_table = (
    washout_sensitivity[
        [
            "weekly_capacity",
            "washout_horizon",
            "evaluation_horizon",
            "value_of_dynamic_optimization",
            "VDO_change_from_no_washout",
            "active_set_changed",
            "product_week_timing_disagreement",
        ]
    ]
    .sort_values(
        [
            "weekly_capacity",
            "washout_horizon",
        ]
    )
)

tables = {
    "table_product_calibration.csv": (
        calibration_table
    ),
    "table_supported_actions.csv": (
        action_table
    ),
    "table_washout_corrected_escalation.csv": (
        escalation_table
    ),
    "table_main_category_comparison.csv": (
        main_comparison
    ),
    "table_capacity_sensitivity.csv": (
        capacity_summary
    ),
    "table_terminal_washout_sensitivity.csv": (
        washout_table
    ),
}

for filename, frame in (
    tables.items()
):
    frame.to_csv(
        PAPER_DIR
        / filename,
        index=False,
    )
    (
        PAPER_DIR
        / filename.replace(
            ".csv",
            ".tex",
        )
    ).write_text(
        frame.to_latex(
            index=False,
            float_format="%.3f",
        ),
        encoding="utf-8",
    )

display(
    main_comparison
)
display(
    washout_table
)
display(
    capacity_summary
)

## 4. Main figures

In [ ]:
primary_comparison = (
    category_comparison.loc[
        category_comparison[
            "grid_name"
        ].eq(
            PRIMARY_GRID
        )
    ]
)

fig, ax = plt.subplots(
    figsize=(
        9,
        5,
    )
)

for capacity, group in (
    primary_comparison.groupby(
        "weekly_capacity",
        observed=True,
    )
):
    ordered = group.sort_values(
        "alpha"
    )
    ax.plot(
        ordered[
            "alpha"
        ],
        ordered[
            "value_of_dynamic_optimization"
        ],
        label=(
            f"B={capacity}"
        ),
    )

ax.axhline(
    0.0,
    linewidth=1.0,
)
ax.axvline(
    MAIN_ALPHA,
    linestyle="--",
    linewidth=1.0,
)
ax.set_xlabel(
    "Contract generosity alpha"
)
ax.set_ylabel(
    "Category VDO"
)
ax.set_title(
    "Dynamic value with six-week terminal washout"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    PAPER_DIR
    / "figure_category_VDO_washout6.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

fig, ax = plt.subplots(
    figsize=(
        8,
        4.8,
    )
)

for capacity, group in (
    washout_sensitivity.groupby(
        "weekly_capacity",
        observed=True,
    )
):
    ordered = group.sort_values(
        "washout_horizon"
    )
    ax.plot(
        ordered[
            "washout_horizon"
        ],
        ordered[
            "value_of_dynamic_optimization"
        ],
        marker="o",
        label=(
            f"B={capacity}"
        ),
    )

ax.set_xlabel(
    "Washout weeks"
)
ax.set_ylabel(
    "Category VDO at alpha=2.24"
)
ax.set_title(
    "Terminal-horizon sensitivity"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    PAPER_DIR
    / "figure_terminal_washout_sensitivity.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

main_calendar = (
    category_calendar.loc[
        category_calendar[
            "grid_name"
        ].eq(
            PRIMARY_GRID
        )
        & category_calendar[
            "weekly_capacity"
        ].eq(
            MAIN_CAPACITY
        )
        & np.isclose(
            category_calendar[
                "alpha"
            ],
            MAIN_ALPHA,
        )
    ]
)

calendar_matrix = (
    main_calendar.pivot_table(
        index=[
            "policy",
            "product_name",
        ],
        columns="week",
        values="discount_depth",
        aggfunc="first",
        fill_value=0.0,
    )
    .sort_index()
)

fig, ax = plt.subplots(
    figsize=(
        10,
        7,
    )
)
image = ax.imshow(
    calendar_matrix.to_numpy(),
    aspect="auto",
)
ax.set_xticks(
    np.arange(
        calendar_matrix.shape[
            1
        ]
    ),
    calendar_matrix.columns,
)
ax.set_yticks(
    np.arange(
        len(
            calendar_matrix
        )
    ),
    [
        f"{policy}: {product}"
        for (
            policy,
            product,
        ) in (
            calendar_matrix.index
        )
    ],
)
ax.set_xlabel(
    "Decision week"
)
ax.set_title(
    "Washout-corrected empirical promotion calendars"
)
fig.colorbar(
    image,
    ax=ax,
    label="Discount depth",
)
fig.tight_layout()
fig.savefig(
    PAPER_DIR
    / "figure_empirical_calendars_washout6.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 5. Export a concise result summary

In [ ]:
main_primary = (
    main_comparison.loc[
        main_comparison[
            "grid_name"
        ].eq(
            PRIMARY_GRID
        )
    ]
)

if main_primary.empty:
    main_vdo = np.nan
    active_change = False
    timing_disagreement = np.nan
else:
    row = main_primary.iloc[
        0
    ]
    main_vdo = float(
        row[
            "value_of_dynamic_optimization"
        ]
    )
    active_change = bool(
        row[
            "active_set_changed"
        ]
    )
    timing_disagreement = int(
        row[
            "product_week_timing_disagreement"
        ]
    )

main_washout = (
    washout_sensitivity.loc[
        washout_sensitivity[
            "weekly_capacity"
        ].eq(
            MAIN_CAPACITY
        )
        & washout_sensitivity[
            "washout_horizon"
        ].eq(
            6
        )
    ]
)

terminal_change = (
    float(
        main_washout.iloc[
            0
        ][
            "VDO_change_from_no_washout"
        ]
    )
    if not main_washout.empty
    else np.nan
)

summary_lines = [
    "# Washout-corrected tiny-paper result summary",
    "",
    "- Decision horizon: 12 weeks",
    "- Primary washout horizon: 6 weeks",
    "- Evaluation horizon: 18 weeks",
    (
        f"- Products: "
        f"{calibration['upc'].nunique()}"
    ),
    (
        "- Corrected promotion-week holdout MAE: "
        f"{holdout.loc[holdout['relative_week'].eq(0), 'absolute_error'].mean():.3f}"
    ),
    (
        "- Post-promotion holdout MAE: "
        f"{holdout.loc[holdout['relative_week'].between(1, 4), 'absolute_error'].mean():.3f}"
    ),
    (
        "- Category VDO at "
        f"alpha={MAIN_ALPHA:.2f}, B={MAIN_CAPACITY}: "
        + (
            f"{main_vdo:.3f}"
            if np.isfinite(
                main_vdo
            )
            else "not available"
        )
    ),
    (
        "- Change in VDO relative to zero washout: "
        + (
            f"{terminal_change:.3f}"
            if np.isfinite(
                terminal_change
            )
            else "not available"
        )
    ),
    (
        "- Dynamic and myopic active sets differ: "
        f"{active_change}"
    ),
    (
        "- Product-week timing disagreements: "
        f"{timing_disagreement}"
    ),
    "",
    "## Interpretation",
    "",
    (
        "- Promotions are selected over 12 weeks but evaluated "
        "through week 18."
    ),
    (
        "- No promotions are permitted during the washout period."
    ),
    (
        "- The main policy uses empirically supported "
        "product-specific actions."
    ),
]

(
    PAPER_DIR
    / "washout_corrected_results_summary.md"
).write_text(
    "\n".join(
        summary_lines
    )
    + "\n",
    encoding="utf-8",
)

print(
    "Generated washout-corrected paper outputs:",
    PAPER_DIR,
)